# Which candidate identifier groups transactions most purely by label?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


# Entity Purity Analysis

**Goal:** Evaluate whether the reconstructed client identifier correctly groups transactions of the same individual.

The dataset contains individual transactions but no explicit `client_id`. Because fraud is typically a property of the client rather than a single transaction, reconstructing a reliable client identifier is crucial. We evaluate candidate identifiers by measuring the purity of their resulting transaction groups.

## Candidate Identifiers

1. **Card Only:** `card1`
2. **Card + Address:** `card1` + `addr1`
3. **Reconstructed Client UID:** `card1` + `addr1`, anchored on account creation date (calculated by subtracting `D1` from the transaction time).

## 1. Purity Evaluation

Evaluating the candidate identifiers using the `compare` function.

In [7]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

from fraud_detection.evaluation.entity_purity import (
    Anchor,
    EntityKey,
    compare,
)


def get_raw_dir():
    import os
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

COLS = ["TransactionID", "TransactionDT", "isFraud", "card1", "addr1", "D1"]
raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).select(COLS).collect()
display(Markdown(f"**Dataset loaded:** {len(df):,} rows"))


**Dataset loaded:** 590,540 rows

In [8]:
card = EntityKey(columns=("card1",), name="card1")
card_addr = EntityKey(columns=("card1", "addr1"), name="card1+addr1")
client = EntityKey(columns=("card1", "addr1"), anchors=(Anchor("D1"),), name="client uid")

comp_df = compare(df, [card, card_addr, client], label="isFraud")[
    ["entity", "entities", "entities_multi", "pure_share_multi", "pure_share_all",
     "singleton_share", "fraud_share_in_touched_groups"]
]

display(Markdown(comp_df.to_pandas().to_markdown(index=False)))


| entity      |   entities |   entities_multi |   pure_share_multi |   pure_share_all |   singleton_share |   fraud_share_in_touched_groups |
|:------------|-----------:|-----------------:|-------------------:|-----------------:|------------------:|--------------------------------:|
| card1       |      13553 |            10109 |             0.848  |           0.8866 |            0.2541 |                          0.249  |
| card1+addr1 |      37531 |            22713 |             0.8958 |           0.937  |            0.3948 |                          0.363  |
| client uid  |     199070 |            83557 |             0.9853 |           0.9938 |            0.5803 |                          0.8489 |

### Identifier Quality Metrics

| Metric | Description |
|--------|-------------|
| `singleton_share` | Percentage of groups containing exactly one transaction. A high value indicates the identifier is too granular |
| `entities_multi` | Number of groups with at least 2 transactions |
| `pure_share_multi` | Percentage of multi-transaction groups that are 100% pure (all fraud or all legitimate). This is the primary metric for identifier quality |
| `fraud_share_in_touched_groups` | Percentage of fraud transactions within groups where at least one fraud occurred (Fraud Infection Strength). True fraudsters exhibit a very high score here |

In [14]:
labels = comp_df['entity'].to_list()
pure_share = comp_df['pure_share_multi'].to_list()
fraud_share = comp_df['fraud_share_in_touched_groups'].to_list()

fig = go.Figure(data=[
    go.Bar(name='Purity (pure_share_multi)', x=labels, y=pure_share, marker_color='#66b3ff', text=[f"{v*100:.1f}%" for v in pure_share], textposition='auto'),
    go.Bar(name='Fraud Strength (fraud_share_in_touched_groups)', x=labels, y=fraud_share, marker_color='#ff9999', text=[f"{v*100:.1f}%" for v in fraud_share], textposition='auto')
])

fig.update_layout(barmode='group', title='Client Key Evolution (Higher is better)',
                  yaxis_title='Percentage', template='plotly_white')
fig.update_yaxes(range=[0, 1.15])
fig.show()
